# DLS Data Analysis and Plotting

This Python notebook can be used to analyze and plot data obtained from dynamic light scattering (DLS).

## Before You Start

Make sure that you are working on a copy of this notebook. Instructions on how to do so can be found [here](https://scribehow.com/shared/Copying_and_Using_a_Python_Notebook_from_GitHub__nPUb96VpS8ibc9BMFKzq1A).

Obtain the SQLite file from the BeNano instrument. This file needs to be in your Google Drive. Make sure that each sample/measurement has a distinct name.

To use this notebook, simply run each section of code (hidden by default) by pressing the "Run cell" button on the left side of each section. Some sections require you to specify certain variables before running the corresponding code. These variables will be explained in detail below.

In [ ]:
# @title Notebook Setup {display-mode: "form"}

# Sets base path on Google Drive for SQLite data.
from google.colab import drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/'


# Imports other necessary packages and functions.
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Checkbox

### Variable Explanation

- **file_path**: Indicates the location and name of your SQLite file. If your SQLite file is in a folder in your Google Drive, add "folder-name/" (no quotation marks) in front of the SQLite file name. Note that this is case and whitespace sensitive.
- **save_to_csv**: Provides option to save table as a CSV file in Google Drive.

In [ ]:
# @title Data Setup {display-mode: "form"}

# Connects to the SQLite file with the data.
file_path = "data.sqlite" # @param {type: "string"}
db = sqlite3.connect(base_path + file_path)
cur = db.cursor()


# Converts relevant databases into a data frame.
df_size_result = pd.read_sql_query("SELECT * FROM SizeResult", db) # Contains main size data.
df_size_record = pd.read_sql_query("SELECT * FROM SizeRecord", db) # Contains run parameters (e.g., temperature).


# Combines data into one data frame.
data = df_size_result
data["Temperature"] = range(0, len(data))

for i in range(0, len(data)):
  data.loc[i, "Temperature"] = df_size_record.loc[df_size_record["ID"] == data.loc[i, "SizeRecordId"], "Temperature"].values[0]


# Provides option to save data frame as a CSV file.
save_to_csv = False # @param {type: "boolean"}

if save_to_csv:
  data.to_csv("/content/drive/MyDrive/full_data.csv")


# Displays combined data.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
data

In [ ]:
# @title Sample Renaming (Optional)

# Asks for old and new name for sample of interest.
old_sample_name = "old_sample" # @param {type: "string"}
new_sample_name = "new_sample" # @param {type: "string"}


# Replaces all instances of old_sample_name with new_sample_name.
for i in range(0, len(data)):
  if data.loc[i, "SizeResultName"][:-2] == old_sample_name:
        data.loc[i, "SizeResultName"] = (new_sample_name + data.loc[i, "SizeResultName"][-2:])


# Displays data frame to verify change took place.
data

In [ ]:
# @title Data Averaging {display-mode: "form"}

# Automatically removes outliers
def reject_outliers(data, m = 2):
  """
  Reject outliers from a list of data points using mean and median.

  Args:
      data: A list of data points.
      m: A multiplier for the median absolute deviation (MAD) to define outliers.

  Returns:
      A list of data points without outliers.
  """
  median = np.median(data)
  mad = np.median(np.abs(data - median)) # Calculate median absolute deviation (MAD)
  upper_bound = median + m * mad
  lower_bound = median - m * mad
  new_data = []
  outlier_index = []

  for i in range(0, len(data)):
    d = data.iloc[i]
    if lower_bound <= d <= upper_bound:
      new_data.append(d)
    else:
      outlier_index.append(i)

  return new_data, outlier_index


# Extracts unique sample names.
group_key_words = data[["SizeResultName", "Temperature"]].copy()
group_key_words['SizeResultName'] = group_key_words["SizeResultName"].str[:-2]
group_key_words = group_key_words.drop_duplicates()


# Calculates average data for each measurement per sample.
data_avg = pd.DataFrame({"sample" : [], "temp": [], "no_meas": [],
                         "avg_part_size": [], "avg_part_size_std_dev": [],
                         "pdi": [], "pdi_std_dev": [],
                         "int_peak_avg": [], "int_std_dev": [], "int_extra_peaks": [],
                         "vol_peak_avg": [], "vol_std_dev": [], "vol_extra_peaks": []})

for i in range(0, len(group_key_words)): # Goes through each unique sample name.
  subset = data[(data["SizeResultName"].str[:-2] == group_key_words["SizeResultName"].iloc[i]) & (data["Temperature"] == group_key_words["Temperature"].iloc[i])]
  contains_inf = False
  outliers = []
  int_first_peak_values = []
  int_extra_peak_values = []
  vol_first_peak_values = []
  vol_extra_peak_values = []

  for j in range(0, len(subset)): # Goes through each measurement per sample.
    avg_part_size_values = subset["ZAvg"]

    for k in range(0, len(avg_part_size_values)): # Checks if at least one of the measurements have a particle size of negative infinity.
      if avg_part_size_values.iloc[k] == float("-inf"):
        contains_inf = True
        break

    pdi_values = subset["PDI"]
    int_data = [entry for entry in subset["IntensityPeakValue"].tolist()[j][2:-2].split(",") if "PeakValue" in entry]

    for k in range(0, len(int_data)): # Extracts intensity distribution peaks from each measurement.
      if k == 0:
        int_first_peak_values = np.append(int_first_peak_values, int_data[k].split(":")[1])
      else:
        int_extra_peak_values = np.append(int_extra_peak_values, int_data[k].split(":")[1])

    int_first_peak_values = pd.to_numeric(int_first_peak_values)
    int_extra_peak_values = pd.to_numeric(int_extra_peak_values)
    vol_data = [entry for entry in subset["VolumePeakValue"].tolist()[j][2:-2].split(",") if "PeakValue" in entry]

    for k in range(0, len(vol_data)): # Extracts volume distribution peaks from each measurement.
      if k == 0:
        vol_first_peak_values = np.append(vol_first_peak_values, vol_data[k].split(":")[1])
      else:
        vol_extra_peak_values = np.append(vol_extra_peak_values, vol_data[k].split(":")[1])

  vol_first_peak_values = pd.to_numeric(vol_first_peak_values)
  vol_extra_peak_values = pd.to_numeric(vol_extra_peak_values)

  if not contains_inf:
      avg_part_size_values, outliers = reject_outliers(avg_part_size_values)
      pdi_values = np.delete(pdi_values, outliers)
      int_first_peak_values = np.delete(int_first_peak_values, outliers)
      vol_first_peak_values = np.delete(vol_first_peak_values, outliers)

  entry = pd.DataFrame([{"sample": group_key_words["SizeResultName"].iloc[i], "temp": group_key_words["Temperature"].iloc[i], "no_meas": len(subset) - len(outliers),
                         "avg_part_size": np.mean(avg_part_size_values) * 10**9, "avg_part_size_std_dev": np.std(avg_part_size_values) * 10**9,
                         "pdi": np.mean(pdi_values), "pdi_std_dev": np.std(pdi_values),
                         "int_peak_avg": np.mean(int_first_peak_values), "int_std_dev": np.std(int_first_peak_values), "int_extra_peaks": int_extra_peak_values,
                         "vol_peak_avg": np.mean(vol_first_peak_values), "vol_std_dev": np.std(vol_first_peak_values), "vol_extra_peaks": vol_extra_peak_values}])
  data_avg = pd.concat([data_avg, entry])


# Provides option to save data frame as a CSV file.
save_to_csv = False # @param {type: "boolean"}

if save_to_csv:
  data_avg.to_csv("/content/drive/MyDrive/averaged_data.csv")


# Displays averaged data.
pd.set_option("display.max_rows", None)
data_avg

### Variable Explanation

- **include_temperature**: Indicates whether samples need to be distinguished by temperature.
- **intensity_min**: Sets minimum size value for intensity distribution graph.
- **intensity_max**: Sets maximum size value for intensity distribution graph.
- **volume_min**: Sets minimum size value for volume distribution graph.
- **volume_max**: Sets maximum size value for volume distribution graph.

In [ ]:
# @title Autocorrelation and Distribution Graphs (Averaged) {display-mode: "form"}

# Indicates if samples need to be differentiated by temperature
include_temperature = True # @param {type: "boolean"}


# Determines the average autocorrelation function from each measurement and their standard deviation.
graph_avg = pd.DataFrame({"time": [], "corr": [], "corr_err": [],
                          "size": [], "int": [], "int_err": [],
                          "vol": [], "vol_err": [],
                          "group": [], "id": []})

for i in range(0, len(group_key_words)): # Goes through each unique sample name.
  subset = data[(data["SizeResultName"].str[:-2] == group_key_words["SizeResultName"].iloc[i]) & (data["Temperature"] == group_key_words["Temperature"].iloc[i])]

  corr_list = []
  corr_avg = []
  corr_std = []

  size_list = []
  int_list = []
  size_avg = []
  int_avg = []
  int_std = []

  vol_list = []
  vol_avg = []
  vol_std = []

  for j in range(0, len(subset)): # Extracts data for each measurement per sample.
    corr_list.append(list(map(float, subset["g2_1"].tolist()[j][1:-1].split(","))))
    size_list.append(list(map(float, subset["SizeGrades"].tolist()[j][1:-1].split(","))))

    try: # Checks if intensity and volume data point exists for given size.
      int_list.append(list(map(float, subset["IntensityDistribution"].tolist()[j][1:-1].split(","))))
      vol_list.append(list(map(float, subset["VolumeDistribution"].tolist()[j][1:-1].split(","))))
    except ValueError:
      size_list.pop()

  corr_df = pd.DataFrame({"corr": corr_list})
  distrib_df = pd.DataFrame({"size": size_list, "int": int_list, "vol": vol_list})

  for j in range(0, len(corr_list[0])): # Goes through each autocorrelation data point index.
    corr_values = []

    for k in range(0, len(corr_df)): # Goes through each measurement.
      corr_values.append(corr_df["corr"][k][j])

    corr_avg.append(np.mean(corr_values))
    corr_std.append(np.std(corr_values))

  for j in range(0, len(size_list[0])): # Goes through each distribution data point index.
    size_values = []
    int_values = []
    vol_values = []

    for k in range(0, len(distrib_df)): # Goes through each measurement.
      size_values.append(distrib_df["size"][k][j])
      int_values.append(distrib_df["int"][k][j])
      vol_values.append(distrib_df["vol"][k][j])

    size_avg.append(np.mean(size_values))
    int_avg.append(np.mean(int_values))
    int_std.append(np.std(int_values))
    vol_avg.append(np.mean(vol_values))
    vol_std.append(np.std(vol_values))

  label = group_key_words["SizeResultName"].iloc[i]

  if include_temperature:
    label += " (" + str(group_key_words["Temperature"].iloc[i]) + "°C)"

  entry = pd.DataFrame({"time": [np.linspace(0, 10**10, len(corr_avg))], "corr": [corr_avg], "corr_err": [corr_std],
                        "size": [size_avg], "int": [int_avg], "int_err": [int_std],
                        "vol": [vol_avg], "vol_err": [vol_std],
                        "group": label, "id": subset["SizeRecordId"].iloc[0]})
  graph_avg = pd.concat([graph_avg, entry])


# Sorts samples by order they were measured.
graph_avg = graph_avg.sort_values(by = "id")


# Makes graphing interactive.
checkboxes = {g: Checkbox(description = g) for g in graph_avg["group"]}
intensity_min = 0 # @param {type: "number"}
intensity_max = 10000 # @param {type: "number"}
volume_min = 0 # @param {type: "number"}
volume_max = 10000 # @param {type: "number"}

def plots(**kwargs):
  fig1, ax1 = plt.subplots() # For autocorrelation function.
  fig2, ax2 = plt.subplots() # For intesity distribution.
  fig3, ax3 = plt.subplots() # For volume distribution.

  for i in range(len(graph_avg)):
    group_name = graph_avg["group"].iloc[i]

    if kwargs.get(group_name, False):
      ax1.plot(graph_avg["time"].iloc[i][:], graph_avg["corr"].iloc[i][:], label = group_name)
      ax1.fill_between(graph_avg["time"].iloc[i][:], np.array(graph_avg["corr"].iloc[i]) - np.array(graph_avg["corr_err"].iloc[i]), np.array(graph_avg["corr"].iloc[i]) + np.array(graph_avg["corr_err"].iloc[i]), alpha = 0.2)

      ax2.plot(graph_avg["size"].iloc[i][:], graph_avg["int"].iloc[i][:], label = group_name)
      ax2.fill_between(graph_avg["size"].iloc[i][:], np.array(graph_avg["int"].iloc[i]) - np.array(graph_avg["int_err"].iloc[i]), np.array(graph_avg["int"].iloc[i]) + np.array(graph_avg["int_err"].iloc[i]), alpha = 0.2)

      ax3.plot(graph_avg["size"].iloc[i][:], graph_avg["vol"].iloc[i][:], label = group_name)
      ax3.fill_between(graph_avg["size"].iloc[i][:], np.array(graph_avg["vol"].iloc[i]) - np.array(graph_avg["vol_err"].iloc[i]), np.array(graph_avg["vol"].iloc[i]) + np.array(graph_avg["vol_err"].iloc[i]), alpha = 0.2)

  ax1.set_xlabel("Time (µs)")
  ax1.set_ylabel("Autocorrelation")
  #ax1.set_xscale("log")
  ax1.legend()

  ax2.set_xlabel("Diameter (nm)")
  ax2.set_ylabel("Intensity (%)")
  ax2.set_xlim(intensity_min, intensity_max)
  ax2.legend()

  ax3.set_xlabel("Diameter (nm)")
  ax3.set_ylabel("Volume (%)")
  ax3.set_xlim(volume_min, volume_max)
  ax3.legend()

  plt.show()

print("Select samples to display:")
interact(plots, **checkboxes);

### Variable Explanation

- **sample_name**: Selects which sample to show the graphs for each measurement for.
- **temperature**: Further filters sample based on temperature, in case some samples have the same name.

In [ ]:
# @title Autocorrelation and Distribution Graphs (Individual) {display-mode: "form"}

# Filters data to specific sample.
sample_name = "sample" # @param {type: "string"}
temperature = 25 # @param {type: "number"}
sample_set = data[(data["SizeResultName"].str.contains(sample_name)) & (data["Temperature"] == temperature)]


# Asks for bounds of the distribution graphs.
intensity_min = 0 # @param {type: "number"}
intensity_max = 10000 # @param {type: "number"}
volume_min = 0 # @param {type: "number"}
volume_max = 10000 # @param {type: "number"}


# Graphs all data from specified sample.
fig1, ax1 = plt.subplots() # For autocorrelation function.
fig2, ax2 = plt.subplots() # For intesity distribution.
fig3, ax3 = plt.subplots() # For volume distribution.
time_data = [np.linspace(0, 10**10, len(corr_avg))]

for i in range(0, len(sample_set)):
  corr_data = list(map(float, sample_set["g2_1"].tolist()[i][1:-1].split(",")))
  size_data = list(map(float, sample_set["SizeGrades"].tolist()[i][1:-1].split(",")))
  int_data = list(map(float, sample_set["IntensityDistribution"].tolist()[i][1:-1].split(",")))
  vol_data = list(map(float, sample_set["VolumeDistribution"].tolist()[i][1:-1].split(",")))

  ax1.plot(time_data[0], corr_data, label = sample_set["SizeResultName"].iloc[i])
  ax2.plot(size_data, int_data, label = sample_set["SizeResultName"].iloc[i])
  ax3.plot(size_data, vol_data, label = sample_set["SizeResultName"].iloc[i])

ax1.legend()
ax2.legend()
ax3.legend()

ax2.set_xlim(intensity_min, intensity_max)
ax3.set_xlim(volume_min, volume_max)

plt.show()

In [ ]:
# @title Bar Graphs (Non-Temperature Experiments) {display-mode: "form"}

# Makes graphing interactive.
checkboxes = {g: Checkbox(description = g) for g in data_avg["sample"]}

def bar_graphs(**kwargs):
  selected = [False] * len(data_avg)
  fig1, ax1 = plt.subplots() # For average particle size.
  fig2, ax2 = plt.subplots() # For polydispersity index (PDI).
  fig3, ax3 = plt.subplots() # For first peak of intensity distribution.
  fig4, ax4 = plt.subplots() # For first peak of volume distribution.

  for i in range(len(data_avg)):
    group_name = data_avg["sample"].iloc[i]

    if kwargs.get(group_name, False):
      selected[i] = True

  ax1.bar(data_avg['sample'][selected], data_avg["avg_part_size"][selected], yerr = data_avg["avg_part_size_std_dev"][selected], capsize = 2)
  ax1.set_xlabel("Sample")
  ax1.set_ylabel("Average Particle Size (nm)")

  ax2.bar(data_avg['sample'][selected], data_avg["pdi"][selected], yerr = data_avg["pdi_std_dev"][selected], capsize = 2)
  ax2.set_xlabel("Sample")
  ax2.set_ylabel("Polydispersity Index")

  ax3.bar(data_avg['sample'][selected], data_avg["int_peak_avg"][selected], yerr = data_avg["int_std_dev"][selected], capsize = 2)
  ax3.set_xlabel("Sample")
  ax3.set_ylabel("Intensity Distribution Peak (nm)")
  ax3.set_title("First Peak in Intensity Distribution")

  ax4.bar(data_avg['sample'][selected], data_avg["vol_peak_avg"][selected], yerr = data_avg["vol_std_dev"][selected], capsize = 2)
  ax4.set_xlabel("Sample")
  ax4.set_ylabel("Volume Distribution Peak (nm)")
  ax4.set_title("First Peak in Volume Distribution")

  plt.show()

print("Select samples to display:")
interact(bar_graphs, **checkboxes);